In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import os

data_path = "~/Desktop/MetaMath/NCESdata.csv"
df = pd.read_csv(data_path)
df.columns = [c.strip() for c in df.columns]
for col in ["BS", "MS", "PHD"]:
    df[col] = df[col].astype(str).str.replace(",", "", regex=False).str.strip()
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df = df.dropna(subset=["Year", "BS", "MS", "PHD"]).copy()
df["Year"] = df["Year"].astype(int)
df = df.sort_values("Year").reset_index(drop=True)

years = df["Year"].to_numpy()
MS = df["MS"].to_numpy(float)
PHD = df["PHD"].to_numpy(float)

def safe_div(a, b):
    a = np.asarray(a, dtype=float)
    if np.isscalar(b):
        return a / b if b != 0 else np.zeros_like(a)
    b = np.asarray(b, dtype=float)
    out = np.zeros_like(a, dtype=float)
    mask = b != 0
    out[mask] = a[mask] / b[mask]
    return out

def simulate_unconstrained(G, params, F0, P0):
    gG = params["gG"]
    aP = params["aP"]
    aF = params["aF"]
    pGF = params["pGF"]
    pGP = params["pGP"]
    pPF_max = params["pPF_max"]
    KF = params["KF"]
    alphaF = params["alphaF"]

    n = len(G)
    P = np.zeros(n, dtype=float)
    F = np.zeros(n, dtype=float)
    P[0], F[0] = P0, F0
    grad_flow = gG * G

    for t in range(n - 1):
        ppf = pPF_max / (1.0 + alphaF * (F[t] / KF))
        P[t + 1] = (1.0 - ppf - aP) * P[t] + pGP * grad_flow[t]
        F[t + 1] = (1.0 - aF) * F[t] + pGF * grad_flow[t] + ppf * P[t]
        P[t + 1] = max(P[t + 1], 0.0)
        F[t + 1] = max(F[t + 1], 0.0)
    return P, F

def simulate_vacancy_limited(G, params, F0, P0):
    gG = params["gG"]
    aP = params["aP"]
    aF = params["aF"]
    pGF = params["pGF"]
    pGP = params["pGP"]
    pPF_max = params["pPF_max"]
    KF = params["KF"]
    alphaF = params["alphaF"]

    n = len(G)
    P = np.zeros(n, dtype=float)
    F = np.zeros(n, dtype=float)
    vacancies = np.zeros(n, dtype=float)
    cand_dir = np.zeros(n, dtype=float)
    cand_post = np.zeros(n, dtype=float)
    hires = np.zeros(n, dtype=float)
    hire_post = np.zeros(n, dtype=float)

    P[0], F[0] = P0, F0
    grad_flow = gG * G

    for t in range(n - 1):
        vacancies[t] = aF * F[t]
        cand_dir[t] = pGF * grad_flow[t]
        cand_post[t] = (pPF_max / (1.0 + alphaF * (F[t] / KF))) * P[t]
        total = cand_dir[t] + cand_post[t]
        hires[t] = min(vacancies[t], total)
        if total > 0:
            hire_post[t] = hires[t] * cand_post[t] / total

        P[t + 1] = (1.0 - aP) * P[t] + pGP * grad_flow[t] - hire_post[t]
        F[t + 1] = F[t] + hires[t] - vacancies[t]
        P[t + 1] = max(P[t + 1], 0.0)
        F[t + 1] = max(F[t + 1], 0.0)

    flows = {
        "vacancies": vacancies,
        "cand_dir": cand_dir,
        "cand_post": cand_post,
        "hires": hires,
        "hire_post": hire_post,
    }
    return P, F, flows

params = {
    "gG": 0.17,
    "aP": 0.25,
    "aF": 0.03,
    "pGF": 0.08,
    "pGP": 0.45,
    "pPF_max": 0.18,
    "KF": 4000.0,
    "alphaF": 1.0,
}
F0, P0 = 4000.0, 1500.0
gG = params["gG"]
G = safe_div(MS + PHD, gG)

scales = {
    "Baseline degree inflow ": 1.0,
    "Upscaled degree inflow ": 1.3,
    "Downscaled degree inflow ": 0.7,
}

uncon = {}
vac = {}
for label, s in scales.items():
    uncon[label] = simulate_unconstrained(G * s, params, F0, P0)
    vac[label] = simulate_vacancy_limited(G * s, params, F0, P0)

outdir = "~/Desktop/MetaMath/metamath_figures"
os.makedirs(outdir, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "axes.grid": False,
    "font.size": 11,
})

def savefig(name):
    path = os.path.join(outdir, name)
    plt.tight_layout()
    plt.savefig(path, dpi=300)
    return path

saved = {}

# Figure 1
plt.figure(figsize=(7.2, 4.2))
for label in scales:
    P, F = uncon[label]
    plt.plot(years, safe_div(F, F[0]), label=label)
plt.xlabel("Year")
plt.ylabel("Faculty stock index ")
#plt.title("Faculty trajectories across degree inflow scenarios (unconstrained)")
plt.legend()
saved["fig01_faculty_unconstrained_index.png"] = savefig("fig01_faculty_unconstrained_index.png")
plt.show()

# Figure 2
plt.figure(figsize=(7.2, 4.2))
for label, s in scales.items():
    P, F = uncon[label]
    openings = params["aF"] * F
    ppf = params["pPF_max"] / (1.0 + params["alphaF"] * (F / params["KF"]))
    candidates = params["pGF"] * (params["gG"] * (G * s)) + ppf * P
    pressure = safe_div(candidates, openings)
    plt.plot(years, pressure, label=label)
plt.xlabel("Year")
plt.ylabel("Candidates per faculty opening")
#plt.title("Faculty market pressure across degree inflow scenarios (unconstrained)")
plt.legend()
saved["fig02_market_pressure_unconstrained.png"] = savefig("fig02_market_pressure_unconstrained.png")
plt.show()

# Figure 3
plt.figure(figsize=(7.2, 4.2))
for label in scales:
    P, _, _ = vac[label]
    plt.plot(years, safe_div(P, P[0]), label=label)
plt.xlabel("Year")
plt.ylabel("Postdoctoral stock index")
#plt.title("Postdoctoral trajectories across degree inflow scenarios (vacancy limited)")
plt.legend()
saved["fig03_postdoc_vacancy_limited_index.png"] = savefig("fig03_postdoc_vacancy_limited_index.png")
plt.show()

# Figure 4
plt.figure(figsize=(7.2, 4.2))
for label in scales:
    _, _, flows = vac[label]
    cand = flows["cand_dir"][:-1] + flows["cand_post"][:-1]
    pressure = safe_div(cand, flows["vacancies"][:-1])
    plt.plot(years[:-1], pressure, label=label)
plt.xlabel("Year")
plt.ylabel("Candidates per faculty opening")
#plt.title("Competition intensity for faculty positions (vacancy limited)")
plt.legend()
saved["fig04_competition_intensity_vacancy_limited.png"] = savefig("fig04_competition_intensity_vacancy_limited.png")
plt.show()

# Figure 5
plt.figure(figsize=(7.2, 4.2))
styles = [
    {"linestyle": "-", "marker": "o", "markevery": 6, "linewidth": 2.2},
    {"linestyle": "--", "marker": "s", "markevery": 6, "linewidth": 2.2},
    {"linestyle": ":", "marker": "^", "markevery": 6, "linewidth": 2.6},
]
for k, label in enumerate(scales):
    _, _, flows = vac[label]
    share = safe_div(flows["hire_post"][:-1], flows["hires"][:-1])
    plt.plot(years[:-1], share, label=label, **styles[k])
plt.ylim(0, 1)
plt.xlabel("Year")
plt.ylabel("Fraction of hires from postdocs")
#plt.title("Composition of faculty hires across degree inflow scenarios")
plt.legend()
saved["fig05_hire_composition_postdoc_share.png"] = savefig("fig05_hire_composition_postdoc_share.png")
plt.show()

# Figure 6 projections
def extend_constant(series, n_add):
    if n_add <= 0:
        return series.copy()
    return np.concatenate([series, np.repeat(series[-1], n_add)])

def fit_linear(series, years_arr, start_year):
    mask = years_arr >= start_year
    x = years_arr[mask].astype(float)
    y = series[mask].astype(float)
    return np.polyfit(x, y, 1)

end_year = 2030
proj_years = np.arange(int(years[0]), int(end_year) + 1)
n_add = len(proj_years) - len(years)

MS_const = extend_constant(MS, n_add)
PHD_const = extend_constant(PHD, n_add)
G_const = safe_div(MS_const + PHD_const, gG)

coef_MS = fit_linear(MS, years, 2005)
coef_PHD = fit_linear(PHD, years, 2005)
x_future = proj_years[len(years):].astype(float)
MS_future = np.maximum(coef_MS[0] * x_future + coef_MS[1], 1.0)
PHD_future = np.maximum(coef_PHD[0] * x_future + coef_PHD[1], 1.0)
MS_trend = np.concatenate([MS, MS_future])
PHD_trend = np.concatenate([PHD, PHD_future])
G_trend = safe_div(MS_trend + PHD_trend, gG)

P_const, _, _ = simulate_vacancy_limited(G_const, params, F0, P0)
P_trend, _, _ = simulate_vacancy_limited(G_trend, params, F0, P0)

idx_last = len(years) - 1
base_const = P_const[idx_last]
base_trend = P_trend[idx_last]

plt.figure(figsize=(7.2, 4.2))
plt.plot(proj_years, safe_div(P_const, base_const), label="Constant degree inflow after 2017", linewidth=2.2)
plt.plot(proj_years, safe_div(P_trend, base_trend), label="Linear degree trend after 2017", linewidth=2.2)
plt.axvline(years[idx_last], linestyle="--", label="End of observed data")
plt.xlabel("Year")
plt.ylabel("Postdoctoral stock index")
#plt.title("Postdoctoral projections under alternative degree assumptions (vacancy limited)")
plt.legend()
saved["fig06_postdoc_projection_index.png"] = savefig("fig06_postdoc_projection_index.png")
plt.show()

# Sensitivity grids
aF_grid = np.linspace(0.015, 0.06, 13)
KF_grid = np.linspace(2500.0, 8000.0, 12)

Z = np.zeros((len(aF_grid), len(KF_grid)), dtype=float)
Tmat = np.full_like(Z, np.nan, dtype=float)
threshold = 3.0

for i, aF in enumerate(aF_grid):
    for j, KF in enumerate(KF_grid):
        p2 = dict(params)
        p2["aF"] = float(aF)
        p2["KF"] = float(KF)
        P, F, _ = simulate_vacancy_limited(G, p2, F0, P0)
        Z[i, j] = P[-1] / F[-1] if F[-1] > 0 else np.nan
        ratio_pf = safe_div(P, F)
        hit = np.where(ratio_pf >= threshold)[0]
        if len(hit) > 0:
            Tmat[i, j] = float(int(years[int(hit[0])]))

# Figure 7 heatmap terminal ratio
plt.figure(figsize=(7.2, 4.6))
im = plt.imshow(
    Z,
    origin="lower",
    aspect="auto",
    extent=[KF_grid[0] / 1e3, KF_grid[-1] / 1e3, aF_grid[0] * 1e2, aF_grid[-1] * 1e2],
    cmap="viridis",
)
plt.xlabel("Faculty capacity $K_F$ (×10³)")
plt.ylabel("Faculty exit rate $a^F$ (×10⁻²)")
#plt.title("Terminal postdoc to faculty ratio")
cb = plt.colorbar(im)
cb.ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
cb.update_ticks()
saved["fig07_heatmap_terminal_ratio.png"] = savefig("fig07_heatmap_terminal_ratio.png")
plt.show()

# Figure 8 3D contour
X, Y = np.meshgrid(KF_grid / 1e3, aF_grid * 1e2)
fig = plt.figure(figsize=(7.2, 5.2))
ax = fig.add_subplot(111, projection="3d")
ax.contour3D(X, Y, Z, 18)

ax.set_xlabel("Faculty capacity $K_F$ (×10³)")
ax.set_ylabel("Faculty exit rate $a^F$ (×10⁻²)")
ax.set_zlabel("Terminal postdoc to faculty ratio")
#ax.set_title("3D contour of terminal postdoc to faculty ratio")
plt.tight_layout()
path8 = os.path.join(outdir, "fig08_contour3d_terminal_ratio.png")
plt.savefig(path8, dpi=300)
saved["fig08_contour3d_terminal_ratio.png"] = path8
plt.show()

# Figure 9 discrete heatmap first year threshold
year_min = int(years.min())
year_max = int(years.max())
year_ticks = np.arange(year_min, year_max + 1, 5)

bounds = np.arange(year_min - 0.5, year_max + 1.5, 1.0)
cmap9 = plt.get_cmap("tab20", len(bounds) - 1)
norm9 = mcolors.BoundaryNorm(bounds, cmap9.N)

plt.figure(figsize=(7.2, 4.6))
im2 = plt.imshow(
    Tmat,
    origin="lower",
    aspect="auto",
    extent=[KF_grid[0] / 1e3, KF_grid[-1] / 1e3, aF_grid[0] * 1e2, aF_grid[-1] * 1e2],
    cmap=cmap9,
    norm=norm9,
)
#plt.xlabel(r'$\alpha^F$')
#plt.ylabel(r'$F_{\mathrm{final}}$')

plt.xlabel("Faculty capacity $K_F$ (×10³)")
plt.ylabel("Faculty exit rate $a^F$ (×10⁻²)")
#plt.title("First year when postdoc to faculty ratio reaches 3")
cb2 = plt.colorbar(im2, ticks=year_ticks)
cb2.ax.yaxis.set_major_locator(mticker.FixedLocator(year_ticks))
cb2.ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%d"))
saved["fig09_heatmap_first_year_threshold.png"] = savefig("fig09_heatmap_first_year_threshold.png")
plt.show()

saved

